# The Keras Workflow: Configure, Compile, Fit, Evaluate, Predict

In the previous notebook you built a network of four ReLU neurons by hand: a forward pass, a backward pass with the chain rule and a gradient descent loop, all written by hand with NumPy. In this notebook, **Keras** performs that work. You describe the layers, and Keras runs the forward pass, computes every gradient and updates the weights for you.

## Learning Objectives

By the end of this notebook, you should be able to:

- Build a Keras `Sequential` model and calculate the number of parameters of each `Dense` layer.
- Compile and train a regression model with a loss, an optimiser, a batch size and a validation split.
- Plot the learning curves of a training run and evaluate the model on a test set.
- Predict the target for a new example, then save the model and load it again.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Fix the random seeds so every run gives the same numbers
rng = np.random.default_rng(42)
keras.utils.set_random_seed(42)

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## 1. Keras and TensorFlow

[Keras](https://keras.io/about/) is a deep learning library with a high-level Python API: you describe a network as a stack of layers and Keras takes care of the training loop. The heavy computation runs in a **backend**, a numerical library written in C++ and CUDA that can use a GPU. Keras 3 supports three backends: [TensorFlow](https://www.tensorflow.org/), JAX and PyTorch. This course uses TensorFlow.

The cell below prints the versions and the backend in use.

In [ ]:
print(f"Keras version: {keras.__version__}")
print(f"Backend: {keras.config.backend()}")

## 2. The Challenge

Each example has two input features, $x_1$ and $x_2$, both between -3 and 3. The target is a bowl, $y = x_1^2 + x_2^2$, plus some noise. A linear model can only fit a flat plane, so a curved surface like this needs a hidden layer.

The data is split with [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) into a training set, used to fit the weights, and a test set of 100 examples that the model only sees at the end.

In [ ]:
bowl = pd.DataFrame({"x1": rng.uniform(-3, 3, size=500), "x2": rng.uniform(-3, 3, size=500)})
bowl["y"] = bowl["x1"] ** 2 + bowl["x2"] ** 2 + rng.normal(0, 0.5, size=500)

X_train, X_test, y_train, y_test = train_test_split(
    bowl[["x1", "x2"]], bowl["y"], test_size=100, random_state=42
)
print(f"Training set: {X_train.shape}, test set: {X_test.shape}")

In [ ]:
points = plt.scatter(X_train["x1"], X_train["x2"], c=y_train, cmap="viridis", s=15)
plt.colorbar(points, label="y")
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Training data: y grows with the distance from the centre")
plt.gca().set_aspect("equal")
plt.show()

The colour shows the target: low in the centre, where both features are close to 0, and high in the corners. The model receives one row of `X_train` at a time, two numbers, and must predict the colour of that point.

## 3. The Recipe for a Keras Model

Every Keras model is built with five steps:

1. **Configure** the model: the layers, the number of neurons per layer and the activation functions.
2. **Compile** it: the loss to minimise, the optimiser that updates the weights and the metrics to report.
3. **Fit** it to the training data: the number of epochs, the batch size and the share of data kept for validation.
4. **Evaluate** it: the learning curves and the score on the test set.
5. **Predict** with it, and save it for later use.

The steps below follow this order.

### Step 1: Configure the Model

[`keras.Sequential`](https://keras.io/api/models/sequential/) stacks layers in order. [`keras.Input`](https://keras.io/api/layers/core_layers/input/) declares the shape of one example: two features. A [`Dense`](https://keras.io/api/layers/core_layers/dense/) layer connects every input to every neuron, like the hidden layer in the previous notebook. Its argument `units` sets the number of neurons, and `activation` the activation function:

- The **hidden layer** has 4 neurons with a ReLU activation, so it can add up 4 bent planes.
- The **output layer** has a single neuron with no activation, because the target is any real number. This is the usual output for regression.

In [ ]:
model = keras.Sequential(
    [
        keras.Input(shape=(2,)),  # x1 and x2
        keras.layers.Dense(units=4, activation="relu", name="hidden"),
        keras.layers.Dense(units=1, name="output"),  # one number: the prediction
    ]
)
model.summary()

#### Where Do the 12 Parameters Come From?

The hidden layer follows the rule from the previous notebook: one weight per input per neuron, plus one bias per neuron. With 2 inputs and 4 neurons that is $2 \times 4 + 4 = 12$. The output neuron receives the 4 hidden outputs, so it has $4 \times 1 + 1 = 5$ parameters, and the model 17 in total.

Keras has already created these weights with small random values. [`get_weights()`](https://keras.io/api/layers/base_layer/#getweights-method) returns them as arrays: a weight matrix with one row per input and one column per neuron, and one bias per neuron.

In [ ]:
W_hidden, b_hidden = model.get_layer("hidden").get_weights()
pd.DataFrame(W_hidden, index=["x1", "x2"], columns=[f"neuron {j}" for j in range(1, 5)]).round(2)

The biases start at 0. The weights are random so that the four neurons start in different places and learn different bends; if they all started equal, they would receive the same gradients and stay identical.

### Step 2: Compile the Model

[`compile()`](https://keras.io/api/models/model_training_apis/#compile-method) fixes how the model learns. It takes three choices:

- **Loss**: what training minimises. The loss depends on the problem:

| Problem | Output layer | Loss |
|---|---|---|
| Regression | 1 neuron, no activation | `"mse"` |
| Binary classification | 1 neuron, sigmoid | `"binary_crossentropy"` |
| Multiclass classification | 1 neuron per class, softmax | `"sparse_categorical_crossentropy"` |

- **Optimiser**: the algorithm that updates the weights. [Adam](https://keras.io/api/optimizers/adam/) is gradient descent that adapts the step size of each weight during training; it is the usual default. The learning rate plays the same role as in your `train_neuron` function.
- **Metrics**: numbers reported during training but not optimised. The root mean squared error (RMSE) is in the units of $y$, which makes it easier to read than the MSE.

In [ ]:
model.compile(
    loss="mse",
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
)

Calling `compile()` again does not reset the weights. To start training from new random weights, run the configuration cell of step 1 again.

### Step 3: Fit the Model

[`fit()`](https://keras.io/api/models/model_training_apis/#fit-method) runs the training loop of the previous notebook: forward pass, loss, backward pass, update. Three arguments control it:

- `epochs`: how many times training goes through the whole training set.
- `batch_size`: how many examples are used for each update. The previous notebook used all examples for every update; Keras uses small batches (**mini-batch gradient descent**), which gives many cheaper updates per epoch.
- `validation_split`: the share of the training data held back to check the model on examples it does not train on. The last 20% of the rows are used.

In [ ]:
history = model.fit(X_train, y_train, epochs=200, batch_size=32, validation_split=0.2, verbose=0)

n_fit = int(len(X_train) * 0.8)  # the rows left for training after the validation split
print(f"{n_fit} training rows / batch size 32 = {np.ceil(n_fit / 32):.0f} updates per epoch")

With 200 epochs of 10 updates each, the weights were updated 2,000 times. `verbose=0` hides the progress bar; set `verbose=1` to watch the loss after every epoch.

Calling `fit()` again does **not** start training over. It continues from the weights the model has now, so running the cell above a second time trains for another 200 epochs, 400 in total, and the learning curves below would only show the second run. To train from scratch, run the configuration cell of step 1 and the `compile()` cell again first.

### Step 4: Evaluate the Model

`fit()` returns a `History` object. Its `history` attribute holds the loss and the metrics after every epoch, for the training data and, with the prefix `val_`, for the validation data. A DataFrame of these values plots the **learning curves**.

In [ ]:
logs = pd.DataFrame(history.history)
logs[["rmse", "val_rmse"]].plot(xlabel="Epoch", ylabel="RMSE")
plt.title("Learning curves")
plt.show()

logs.tail(3).round(3)

The RMSE falls quickly during the first 50 epochs and then levels off. After 200 epochs it is 0.80 on the training rows and 0.86 on the validation rows. The two curves stay close together, so the model is not overfitting: it performs almost as well on rows it never trained on.

In [ ]:
test_mse, test_rmse = model.evaluate(X_test, y_test, verbose=0)
print(f"Test RMSE: {test_rmse:.3f}")

The test RMSE is 0.774: on examples the model has never seen, a typical prediction is off by about 0.8. The noise added to the data has a standard deviation of 0.5, so no model can get much below 0.5.

### Step 5: Predict with the Model

[`predict()`](https://keras.io/api/models/model_training_apis/#predict-method) expects the same layout as the training data: one row per example and one column per feature. A single new example is therefore a DataFrame with one row, of shape (1, 2), not a list of two numbers.

In [ ]:
new_point = pd.DataFrame({"x1": [1.0], "x2": [2.0]})  # one row: one example
print(f"Shape of the input: {new_point.shape}")

prediction = model.predict(new_point, verbose=0)
print(f"Prediction: {prediction[0, 0]:.2f}, true value without noise: {1.0**2 + 2.0**2:.2f}")

For $x_1 = 1$ and $x_2 = 2$ the model predicts 5.41, while the bowl without noise gives $1 + 4 = 5$. `predict()` returns an array with one row per example and one column per output neuron, so the single value is read with `[0, 0]`.

The left plot below compares the predictions for the test set with the true values. The right plot walks along $x_1$ with $x_2$ fixed at 0, a slice through the middle of the bowl.

In [ ]:
fig, (ax_test, ax_slice) = plt.subplots(1, 2, figsize=(12, 4.5))

y_pred = model.predict(X_test, verbose=0)[:, 0]
ax_test.scatter(y_test, y_pred, alpha=0.6)
ax_test.plot([0, 18], [0, 18], color="black", linestyle="--", label="Perfect prediction")
ax_test.set_xlabel("True y")
ax_test.set_ylabel("Predicted y")
ax_test.set_title("Test set")
ax_test.legend()

x1_line = np.linspace(-3, 3, 200)
slice_x2_0 = pd.DataFrame({"x1": x1_line, "x2": 0.0})  # walk along x1 with x2 fixed at 0
ax_slice.plot(x1_line, x1_line**2, color="#2e9467", linestyle="--", label="True curve y = x1²")
ax_slice.plot(x1_line, model.predict(slice_x2_0, verbose=0)[:, 0], color="black", label="Model")
ax_slice.set_xlabel("x1 (with x2 = 0)")
ax_slice.set_ylabel("y")
ax_slice.set_title("A slice through the bowl")
ax_slice.legend()

fig.tight_layout()
plt.show()

Most test points lie close to the dashed line, so the predictions follow the bowl. The slice shows how: the model's curve is made of straight pieces, like the four hand-built ReLUs in the previous notebook. With only 4 neurons the pieces are coarse: at the bottom of the bowl the model is about 0.9 too low, and near the edges its straight lines miss the parabola.

#### Save and Load the Model

[`model.save()`](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/) writes the layers, the trained weights and the compile settings to a single `.keras` file. [`keras.saving.load_model()`](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/#loadmodel-function) reads it back.

In [ ]:
Path("models").mkdir(exist_ok=True)
model.save("models/bowl_model.keras")

In [ ]:
loaded_model = keras.saving.load_model("models/bowl_model.keras")
same = np.allclose(loaded_model.predict(X_test, verbose=0), model.predict(X_test, verbose=0))
print(f"The loaded model gives the same predictions: {same}")

The loaded model makes exactly the same predictions, so training does not have to be repeated. The folder `models/` is ignored by git, because trained models can be large.

## 4. The Whole Recipe in One Cell

So far each step had its own cell, which makes it easy to inspect the model in between. It also makes it possible to continue training unintentionally from the weights of the last run, because `fit()` continues from the current weights. When you experiment, put the whole recipe in **one cell**: every run of the cell then builds a new model with new random weights and trains it from scratch.

Two calls at the top make each run independent of the ones before:

- [`keras.utils.clear_session()`](https://keras.io/api/utils/config_utils/#clearsession-function) resets the state Keras keeps between models, such as the counter behind automatic layer names (`dense`, `dense_1`, `dense_2`, ...), and frees the memory of models you no longer use.
- `keras.utils.set_random_seed(42)` makes the starting weights the same on every run, so two runs with the same settings give the same result.

In [ ]:
keras.utils.clear_session()  # start from a clean Keras state
keras.utils.set_random_seed(42)  # the same starting weights on every run

# 1. Configure
model = keras.Sequential(
    [
        keras.Input(shape=(2,)),
        keras.layers.Dense(units=4, activation="relu", name="hidden"),
        keras.layers.Dense(units=1, name="output"),
    ]
)

# 2. Compile
model.compile(
    loss="mse",
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
)

# 3. Fit
history = model.fit(X_train, y_train, epochs=200, batch_size=32, validation_split=0.2, verbose=0)

# 4. Evaluate
test_rmse = model.evaluate(X_test, y_test, verbose=0)[1]
print(f"Epochs trained: {len(history.history['loss'])}, test RMSE: {test_rmse:.3f}")

The cell trains for exactly 200 epochs and reaches the same test RMSE as the step-by-step model, 0.774, because the seed gives it the same starting weights. Run the cell a second time and you get 200 epochs and 0.774 again, not 400 epochs. The Try It below uses this one-cell pattern.

## Try It: Give the Network More Neurons

Four neurons can only build the bowl from four bent planes. Build a new model with more neurons in the hidden layer, or a second hidden layer, and train it until the **validation** RMSE falls below 0.6. The test set is kept for the final comparison, so the Try It does not use it.

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)

# 32 neurons give enough bent planes to follow the bowl closely
hidden_layers = [
    keras.layers.Dense(units=32, activation="relu"),
]
n_epochs = 200

model_try = keras.Sequential([keras.Input(shape=(2,)), *hidden_layers, keras.layers.Dense(units=1)])
model_try.compile(
    loss="mse",
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    metrics=[keras.metrics.RootMeanSquaredError(name="rmse")],
)
history_try = model_try.fit(X_train, y_train, epochs=n_epochs, batch_size=32, validation_split=0.2, verbose=0)

In [ ]:
valid_rmse_try = history_try.history["val_rmse"][-1]  # the validation RMSE after the last epoch
print(f"Validation RMSE: {valid_rmse_try:.3f} (4-neuron model: {history.history['val_rmse'][-1]:.3f})")

## Summary

- Keras describes a network as a stack of layers and runs the training loop for you on a backend such as TensorFlow.
- A Keras model is built in five steps: configure, compile, fit, evaluate and predict.
- A `Dense` layer has one weight per input per neuron and one bias per neuron, so 2 inputs and 4 neurons give 12 parameters.
- For regression the output layer is one neuron with no activation and the loss is the MSE; the loss changes with the problem type.
- `fit()` uses mini-batches and can hold back a validation split; its `History` gives the learning curves.
- `predict()` expects one row per example, even for a single example, and a trained model can be saved to a `.keras` file and loaded again.
- `fit()` continues from the current weights. Putting the whole recipe in one cell, after `keras.utils.clear_session()` and a fixed seed, trains a new model on every run.

Next, continue with [03 - Regression with a Feed-Forward Network](03_regression_with_neural_networks.ipynb).

## References

- Keras API: [models](https://keras.io/api/models/), [layers](https://keras.io/api/layers/), [optimisers](https://keras.io/api/optimizers/), [losses](https://keras.io/api/losses/) and [metrics](https://keras.io/api/metrics/).
- Keras guides: [The Sequential model](https://keras.io/guides/sequential_model/) and [Save, serialise and export models](https://keras.io/guides/serialization_and_saving/).
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd edition, chapter 10.
- Chollet, F. and Watson, M. (2025). *Deep Learning with Python*, 3rd edition, chapter 3: Introduction to TensorFlow, PyTorch, JAX and Keras. [Free to read online](https://deeplearningwithpython.io/).
- Google [Machine Learning Glossary](https://developers.google.com/machine-learning/glossary), for any term you want to look up again.